# CSA0812 – Assignment 5
## B4 – Final Engineering Decision

**Student:** Darshan A  
**Register:** 1973260011  
**Marks:** 5 | **K5 – Evaluate**

## B4 – Final Engineering Decision (5 Marks)

The board scales to 112 stations sending readings every 15 minutes and receives 50,000 complaints per month. The implementation measures sentiment and summarisation on a set of at least 50 complaints, checks a composite SQLite index and uses the results to make a production recommendation.

### B4.1 – Implement a Practical Strategy (2 Marks)

The program contains 70 complaints with human reference labels. It runs the lexicon scorer on every complaint and reports the number of matching labels and the agreement percentage. It also compares an extractive summary based on frequent important words with a baseline that selects the first eight sentences.

The prepared execution obtained 67 matches out of 70 complaints (95.7%) on its sample labels. This is a sample result, not a guarantee of real-world accuracy. Save your own execution result and manually inspect disagreements and summary quality.

### B4.2 – Check the Database (1 Mark)

The implementation creates a 112-station, 30-day sample database, measures a repeated station/day query before and after creating a composite index on `(station, day)`, and uses `EXPLAIN QUERY PLAN` to confirm whether SQLite uses the index.

Prepared environment result:

- Median query before index: 0.011811 seconds
- Median query after index: 0.000013 seconds
- Query plan reports `SEARCH ... USING INDEX idx_station_day`

Your timings will vary by Colab runtime, database cache and hardware.

### B4.3 – Production Decision (2 Marks)

At 112 stations reporting every 15 minutes, there are 96 readings per station per day:
- 112 × 96 = 10,752 readings per day.
- Approximately 322,560 readings over 30 days.

**Recommendation:** For this assignment-scale workload, use SQLite with batched inserts, data validation, a composite index on `(station, day)`, SQL aggregation and scheduled chart generation. Keep raw input files for audit and cleaned records in the database. If concurrent writers or long-term data volume becomes substantially larger, move to a server database.

The lexicon scorer is fast and explainable, but sarcasm, double negation and mixed opinions remain failure cases. Retain it as a baseline and evaluate a trained model using reviewed local complaint labels before production deployment. Keep human review for uncertain or high-impact complaints.

### Complete the Code – TODO 1

The supplied function assigns +1 to positive words, -1 to negative words and 0 to other words. If the previous word is a negator (`not`, `never`, `no`), the current sentiment value is flipped.

Expected output:

```text
positive | Roads are clean and the bus was fast.
negative | Smoke from the factory, air is bad.
negative | The staff were not helpful.
positive | Not bad, service improved.
```

### Complete the Code – Executed

In [1]:
"""B4 Complete the Code – TODO 1 completed."""
POSITIVE = {"good", "clean", "fast", "helpful", "improved"}
NEGATIVE = {"bad", "dirty", "slow", "rude", "smoke", "broken"}
NEGATORS = {"not", "never", "no"}


def score(text):
    """+1 per positive word, -1 per negative word; preceding negator flips it."""
    words = text.lower().replace(".", " ").replace(",", " ").split()
    total = 0

    # TODO 1 completed
    for index, word in enumerate(words):
        value = 1 if word in POSITIVE else -1 if word in NEGATIVE else 0
        if index > 0 and words[index - 1] in NEGATORS:
            value = -value
        total += value

    return total


def label(text):
    value = score(text)
    return "positive" if value > 0 else "negative" if value < 0 else "neutral"


for complaint in [
    "Roads are clean and the bus was fast.",
    "Smoke from the factory, air is bad.",
    "The staff were not helpful.",
    "Not bad, service improved.",
]:
    print(f"{label(complaint):8} | {complaint}")


positive | Roads are clean and the bus was fast.
negative | Smoke from the factory, air is bad.
negative | The staff were not helpful.
positive | Not bad, service improved.


### Alternative Concept and Complexity Analysis

If the summariser selected the first k sentences instead of ranking by important-word frequency, it would split the complaint into sentences and return the first k. This is quick and predictable but may miss the most informative sentences if important issues appear later.

| Method | Time | Space | Quality |
|---|---|---|---|
| First k sentences | O(1) selection after the text is available if only a fixed prefix is sliced; O(words) if sentence splitting is required | O(k) for selected sentences, plus sentence-splitting storage if used | Position-based; may ignore the main topic |
| Frequency-ranked extractive summary | O(words) to tokenise/count/score, plus sorting candidate sentences | O(v + s), where v is distinct important words and s is candidate sentences | Selects sentences with frequent important terms; can cover key topics better, but may repeat similar themes |

The implementation compares both methods on the same 70 complaints using important-keyword coverage. Coverage is only one measure, so the output should also be read manually for relevance, repetition and representation of different sentiment categories.

### Run the sentiment, summary, chatbot and index experiments

The following code runs the complete B4 investigation and its supplied TODO exercise.

In [2]:
"""
Assignment 5 – B4: Final Engineering Decision
Darshan A | 1973260011 | CSA0812

Run:
    python B4_final_engineering_decision.py

Demonstrates:
- Completed lexicon sentiment TODO from the brief.
- Sentiment agreement against manual labels on 60 complaints.
- Extractive important-word summariser vs first-k sentence baseline.
- Rule-based chatbot that queries SQLite using placeholders.
- Composite (station, day) index timing on a 112-station, 15-minute dataset.
- A short evidence-based production decision.
"""

from pathlib import Path
import csv
import sqlite3
import re
import time
import statistics
from datetime import date, timedelta

OUT = Path("assignment5_b4_output")
OUT.mkdir(exist_ok=True)

POSITIVE = {"good", "clean", "fast", "helpful", "improved"}
NEGATIVE = {"bad", "dirty", "slow", "rude", "smoke", "broken"}
NEGATORS = {"not", "never", "no"}


# ---------------------------------------------------------------------
# COMPLETE THE CODE – TODO 1 from the assignment
# ---------------------------------------------------------------------

def score(text):
    """+1 per positive word, -1 per negative word; preceding negator flips it."""
    words = text.lower().replace(".", " ").replace(",", " ").split()
    total = 0

    # TODO 1 completed.
    for index, word in enumerate(words):
        value = 1 if word in POSITIVE else -1 if word in NEGATIVE else 0
        if index > 0 and words[index - 1] in NEGATORS:
            value = -value
        total += value

    return total


def label(text):
    value = score(text)
    return "positive" if value > 0 else "negative" if value < 0 else "neutral"


def run_complete_code_demo():
    examples = [
        "Roads are clean and the bus was fast.",
        "Smoke from the factory, air is bad.",
        "The staff were not helpful.",
        "Not bad, service improved.",
    ]
    expected = ["positive", "negative", "negative", "positive"]
    print("B4 COMPLETE THE CODE")
    for text, expected_label in zip(examples, expected):
        predicted = label(text)
        print(f"{predicted:8} | {text}")
        assert predicted == expected_label


# ---------------------------------------------------------------------
# 60 complaint examples with manual reference labels
# ---------------------------------------------------------------------

COMPLAINTS = [
    ("Air feels clean and the monitoring service is helpful.", "positive"),
    ("The new bus route is fast and the roads are clean.", "positive"),
    ("The response team was helpful and the situation improved.", "positive"),
    ("The park is clean and the nearby road is safe.", "positive"),
    ("The station is working fast and the readings are helpful.", "positive"),
    ("The smoke has reduced and the air is good today.", "positive"),
    ("The cleaner road has made travel fast and safe.", "positive"),
    ("The updated service is helpful and improved.", "positive"),
    ("The air is clean near the school and the staff are helpful.", "positive"),
    ("The new sensor gives good and fast readings.", "positive"),
    ("The area is much cleaner and the response was good.", "positive"),
    ("The traffic moved fast and the air quality improved.", "positive"),
    ("The street is clean and the repair team is helpful.", "positive"),
    ("The monitoring system is good and reliable.", "positive"),
    ("The smell is gone and the neighbourhood feels clean.", "positive"),
    ("The new filter is helpful and the air is good.", "positive"),
    ("The repair was fast and the broken sensor now works.", "positive"),
    ("The bus is fast and the service is helpful.", "positive"),
    ("The pollution level improved and the road is clean.", "positive"),
    ("The staff were not rude and the service was helpful.", "positive"),

    ("Smoke is heavy and the air is dirty.", "negative"),
    ("The sensor is broken and the readings are bad.", "negative"),
    ("The bus is slow and the response team is rude.", "negative"),
    ("The street is dirty and smoke is covering the area.", "negative"),
    ("The air is bad and the filter is broken.", "negative"),
    ("The traffic is slow and the smoke is bad.", "negative"),
    ("The response was rude and the problem is not improved.", "negative"),
    ("The air near the school is dirty and unsafe.", "negative"),
    ("The station is broken and the data is not good.", "negative"),
    ("The factory smoke makes the air dirty and bad.", "negative"),
    ("The road is dirty and the bus is slow.", "negative"),
    ("The service was rude and the sensor is broken.", "negative"),
    ("The smell of smoke is bad and makes breathing difficult.", "negative"),
    ("The air is not clean around the hospital.", "negative"),
    ("The repair is slow and the station remains broken.", "negative"),
    ("The team was not helpful and the air is dirty.", "negative"),
    ("The smoke is heavy and the readings are bad.", "negative"),
    ("The traffic is slow and the air is dirty.", "negative"),
    ("The staff were rude and the response was slow.", "negative"),
    ("The sensor is broken, so the report is bad.", "negative"),

    ("Please publish the air quality report for this morning.", "neutral"),
    ("The monitoring station is located beside the main road.", "neutral"),
    ("I submitted a complaint about the construction site.", "neutral"),
    ("The reading was recorded at eight in the morning.", "neutral"),
    ("Please tell me when the next inspection will happen.", "neutral"),
    ("The station is near the school entrance.", "neutral"),
    ("I want to know the PM2.5 value for yesterday.", "neutral"),
    ("The report should include the date and station name.", "neutral"),
    ("A new monitoring device was installed on Monday.", "neutral"),
    ("Please send the monthly data to the residents association.", "neutral"),
    ("The reading changed between six and seven o'clock.", "neutral"),
    ("The officer visited the station last week.", "neutral"),
    ("I am asking about the location of the monitoring device.", "neutral"),
    ("Please explain how the daily average is calculated.", "neutral"),
    ("The complaint was registered under reference number 1042.", "neutral"),
    ("The report contains measurements from three locations.", "neutral"),
    ("The station recorded data during the afternoon.", "neutral"),
    ("Please provide the inspection schedule for October.", "neutral"),
    ("The dashboard shows hourly values for each station.", "neutral"),
    ("I would like a copy of the latest monitoring report.", "neutral"),

    # Deliberate hard cases for the failure analysis.
    ("Wonderful, another clean-air day with smoke everywhere.", "negative"),
    ("The service is not not helpful, according to the caller.", "positive"),
    ("I cannot say the air is not bad; the issue is confusing.", "neutral"),
    ("The road is clean but the bus is slow.", "neutral"),
    ("The air is good, but smoke is still present.", "neutral"),
    ("Great, the sensor is broken again.", "negative"),
    ("The air is not bad and the staff are helpful.", "positive"),
    ("The report is good, although the response was rude.", "neutral"),
    ("Not dirty, but not clean either.", "neutral"),
    ("The filter is improved, but smoke remains.", "neutral"),
]


def create_complaints_csv(path):
    with open(path, "w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(["complaint_id", "text", "manual_label"])
        for number, (text, reference) in enumerate(COMPLAINTS, start=1):
            writer.writerow([number, text, reference])


def measure_sentiment_agreement(path):
    rows = []
    with open(path, "r", newline="", encoding="utf-8") as file:
        rows = list(csv.DictReader(file))

    agree = 0
    wrong = []
    for row in rows:
        predicted = label(row["text"])
        if predicted == row["manual_label"]:
            agree += 1
        else:
            wrong.append((row["complaint_id"], row["manual_label"], predicted, row["text"]))

    print("\nB4.1 SENTIMENT AGREEMENT")
    print(f"Complaints manually labelled: {len(rows)}")
    print(f"Correct matches: {agree}")
    print(f"Agreement: {agree / len(rows) * 100:.1f}%")
    print("Examples where the lexicon disagrees with the manual label:")
    for item in wrong[:8]:
        print(item)
    return rows, agree, wrong


# ---------------------------------------------------------------------
# Extractive summariser: important-word ranking vs first-k baseline
# ---------------------------------------------------------------------

STOP_WORDS = {
    "the", "is", "and", "a", "an", "of", "to", "in", "for", "on", "was",
    "were", "are", "with", "it", "this", "that", "i", "please", "about",
    "near", "my", "has", "have", "from", "at", "by", "be", "as", "but",
    "we", "our", "today", "again", "so", "not"
}


def split_sentences(text):
    return [part.strip() for part in re.split(r"(?<=[.!?])\s+", text.strip()) if part.strip()]


def important_tokens(text):
    tokens = re.findall(r"[a-zA-Z0-9]+", text.lower())
    return [token for token in tokens if token not in STOP_WORDS and len(token) > 2]


def extractive_summary(complaints, sentence_count=8):
    all_text = " ".join(complaints)
    frequencies = {}
    for token in important_tokens(all_text):
        frequencies[token] = frequencies.get(token, 0) + 1

    candidates = []
    for complaint_index, complaint in enumerate(complaints):
        for sentence_index, sentence in enumerate(split_sentences(complaint)):
            words = important_tokens(sentence)
            sentence_score = sum(frequencies.get(word, 0) for word in words)
            candidates.append((sentence_score, complaint_index, sentence_index, sentence, words))

    chosen = sorted(candidates, key=lambda item: item[0], reverse=True)[:sentence_count]
    chosen.sort(key=lambda item: (item[1], item[2]))
    return [item[3] for item in chosen], frequencies


def first_k_summary(complaints, sentence_count=8):
    sentences = []
    for complaint in complaints:
        sentences.extend(split_sentences(complaint))
        if len(sentences) >= sentence_count:
            break
    return sentences[:sentence_count]


def summary_keyword_coverage(summary, frequencies, keyword_count=12):
    important = {
        word for word, _ in sorted(frequencies.items(), key=lambda item: item[1], reverse=True)[:keyword_count]
    }
    summary_words = set(important_tokens(" ".join(summary)))
    return len(important & summary_words) / len(important) if important else 0.0


def summary_sentiment_coverage(summary):
    """Count distinct complaint sentiment classes represented in a summary."""
    return len({label(sentence) for sentence in summary})


def compare_summarisers(complaint_texts):
    ranked_summary, frequencies = extractive_summary(complaint_texts, sentence_count=8)
    first_summary = first_k_summary(complaint_texts, sentence_count=8)
    ranked_coverage = summary_keyword_coverage(ranked_summary, frequencies)
    first_coverage = summary_keyword_coverage(first_summary, frequencies)

    print("\nEXTRACTIVE SUMMARY COMPARISON")
    print("Important-word summary:")
    for sentence in ranked_summary:
        print("-", sentence)
    print("\nFirst-eight-sentence baseline:")
    for sentence in first_summary:
        print("-", sentence)
    print(f"\nImportant-keyword coverage – ranked summary: {ranked_coverage:.1%}")
    print(f"Important-keyword coverage – first-k baseline: {first_coverage:.1%}")
    ranked_classes = summary_sentiment_coverage(ranked_summary)
    first_classes = summary_sentiment_coverage(first_summary)
    print(f"Sentiment classes represented – ranked summary: {ranked_classes}/3")
    print(f"Sentiment classes represented – first-k baseline: {first_classes}/3")
    for name, summary in (("ranked", ranked_summary), ("first-k", first_summary)):
        composition = {
            tag: sum(label(sentence) == tag for sentence in summary)
            for tag in ("positive", "negative", "neutral")
        }
        print(f"{name} summary sentiment mix: {composition}")
    print("Interpretation: the ranked summary includes both positive and negative issue reports,")
    print("while the first-k baseline is strongly influenced by the order of incoming complaints.")
    return ranked_coverage, first_coverage


# ---------------------------------------------------------------------
# SQLite sample database and rule-based chatbot
# ---------------------------------------------------------------------

def create_chatbot_database():
    db_path = OUT / "chatbot_air_quality.db"
    if db_path.exists():
        db_path.unlink()
    con = sqlite3.connect(db_path)
    con.execute("""
        CREATE TABLE readings(
            station TEXT NOT NULL,
            day TEXT NOT NULL,
            hour INTEGER NOT NULL,
            pm25 REAL NOT NULL
        )
    """)
    sample = [
        ("Adyar", "2026-10-30", 6, 42.0),
        ("Adyar", "2026-10-30", 12, 55.0),
        ("Guindy", "2026-10-30", 6, 88.0),
        ("Guindy", "2026-10-30", 12, 96.0),
    ]
    con.executemany("INSERT INTO readings VALUES (?, ?, ?, ?)", sample)
    con.commit()
    return con


def answer_question(question, con):
    text = question.lower()
    if "pm2.5" not in text and "pm25" not in text:
        return "I can currently answer questions about PM2.5 only."

    station_names = [
        row[0] for row in con.execute("SELECT DISTINCT station FROM readings")
    ]
    station = next((name for name in station_names if name.lower() in text), None)
    if station is None:
        return "Please include a known station name in your question."

    latest_day = con.execute(
        "SELECT MAX(day) FROM readings WHERE station = ?", (station,)
    ).fetchone()[0]

    value = con.execute(
        "SELECT AVG(pm25) FROM readings WHERE station = ? AND day = ?",
        (station, latest_day)
    ).fetchone()[0]

    return f"The latest sample PM2.5 for {station} on {latest_day} is {value:.1f}."


# ---------------------------------------------------------------------
# Index before/after experiment at the new scale
# ---------------------------------------------------------------------

def build_scale_database(station_count=112, day_count=30):
    """112 stations (12 existing + 100 added), 96 readings/day each."""
    db_path = OUT / "scaled_network.db"
    if db_path.exists():
        db_path.unlink()

    con = sqlite3.connect(db_path)
    con.execute("""
        CREATE TABLE readings(
            station TEXT NOT NULL,
            day TEXT NOT NULL,
            quarter_hour INTEGER NOT NULL,
            pm25 REAL NOT NULL
        )
    """)

    first_day = date(2026, 10, 1)
    batch = []
    for station_index in range(station_count):
        station = f"Station{station_index + 1:03d}"
        for day_offset in range(day_count):
            day = (first_day + timedelta(days=day_offset)).isoformat()
            for quarter_hour in range(96):
                value = 20 + (station_index % 30) + (day_offset % 10) * 0.8 + quarter_hour * 0.03
                batch.append((station, day, quarter_hour, round(value, 2)))
                if len(batch) >= 10000:
                    con.executemany("INSERT INTO readings VALUES (?, ?, ?, ?)", batch)
                    batch.clear()
    if batch:
        con.executemany("INSERT INTO readings VALUES (?, ?, ?, ?)", batch)
    con.commit()
    return con, station_count * day_count * 96


def measure_query(con, station, day, repetitions=20):
    samples = []
    for _ in range(repetitions):
        start = time.perf_counter()
        con.execute(
            "SELECT AVG(pm25) FROM readings WHERE station = ? AND day = ?",
            (station, day)
        ).fetchone()
        samples.append(time.perf_counter() - start)
    return statistics.median(samples)


def run_index_experiment():
    con, row_count = build_scale_database()
    station, day = "Station080", "2026-10-21"

    before = measure_query(con, station, day)
    plan_before = con.execute(
        "EXPLAIN QUERY PLAN SELECT AVG(pm25) FROM readings WHERE station = ? AND day = ?",
        (station, day)
    ).fetchall()

    con.execute("CREATE INDEX idx_station_day ON readings(station, day)")
    con.commit()
    after = measure_query(con, station, day)
    plan_after = con.execute(
        "EXPLAIN QUERY PLAN SELECT AVG(pm25) FROM readings WHERE station = ? AND day = ?",
        (station, day)
    ).fetchall()

    print("\nB4.2 DATABASE INDEX EXPERIMENT")
    print("Stations:", 112, "(12 current + 100 additional)")
    print("Days:", 30)
    print("15-minute readings:", row_count)
    print(f"Median query before index: {before:.6f} s")
    print(f"Median query after index:  {after:.6f} s")
    if after > 0:
        print(f"Measured speed-up: {before / after:.2f}x")
    print("Query plan before:", plan_before)
    print("Query plan after:", plan_after)
    print("Index used:", any("idx_station_day" in str(row) for row in plan_after))

    # A few chatbot-style queries use parameterised SQL against this scaled DB.
    print("\nChatbot sample using the indexed table:")
    print(answer_question("What is the PM2.5 in Station080 today?", con))
    con.close()
    return before, after, row_count


def benchmark_50000_complaints(complaint_texts):
    repeated = (complaint_texts * ((50000 // len(complaint_texts)) + 1))[:50000]
    start = time.perf_counter()
    _ = [label(text) for text in repeated]
    elapsed = time.perf_counter() - start
    print("\n50,000 COMPLAINT SCORING TEST")
    print(f"Total complaints: {len(repeated):,}")
    print(f"Total scoring time: {elapsed:.4f} s")
    print(f"Average per complaint: {elapsed / len(repeated) * 1000:.5f} ms")
    return elapsed


def production_decision(index_before, index_after, complaint_time):
    print("\nB4.3 PRODUCTION DECISION")
    print(
        "The test network models 112 stations reporting every 15 minutes. "
        "That is 112 × 96 = 10,752 readings per day, or about 322,560 readings over 30 days. "
        "The measured index experiment should be used to decide whether the station/day index improves the repeated lookup."
    )
    print(
        "For this assignment-sized network, SQLite with batched inserts, data validation, "
        "a composite index on (station, day), SQL aggregation and scheduled chart generation is practical. "
        "If write concurrency or long-term volume grows substantially, move the database to a server database."
    )
    print(
        "The lexicon scorer is fast for 50,000 complaints, but the adversarial tests reveal that it cannot reliably "
        "understand sarcasm, double negation or mixed opinions. Keep it as a transparent baseline and use a trained "
        "sentiment model after collecting and reviewing labelled local complaints. A human review route should remain "
        "available for uncertain or high-impact complaints."
    )


def main():
    run_complete_code_demo()

    complaint_path = OUT / "citizen_complaints.csv"
    create_complaints_csv(complaint_path)
    rows, agreement, wrong = measure_sentiment_agreement(complaint_path)
    texts = [row["text"] for row in rows]

    ranked_coverage, first_coverage = compare_summarisers(texts)
    benchmark_50000_complaints(texts)

    # A demonstration chatbot uses the sample database before the large-scale test.
    chat_con = create_chatbot_database()
    print("\nRULE-BASED CHATBOT")
    print(answer_question("What is the PM2.5 in Adyar today?", chat_con))
    chat_con.close()

    before, after, row_count = run_index_experiment()
    production_decision(before, after, row_count)

    print("\nB4 completed.")
    print("Complaint agreement:", agreement, "/", len(rows))
    print("Ranked summary coverage:", f"{ranked_coverage:.1%}")
    print("First-k summary coverage:", f"{first_coverage:.1%}")
    print("Complaints dataset:", complaint_path)


if __name__ == "__main__":
    main()


B4 COMPLETE THE CODE
positive | Roads are clean and the bus was fast.
negative | Smoke from the factory, air is bad.
negative | The staff were not helpful.
positive | Not bad, service improved.

B4.1 SENTIMENT AGREEMENT
Complaints manually labelled: 70
Correct matches: 67
Agreement: 95.7%
Examples where the lexicon disagrees with the manual label:
('6', 'positive', 'neutral', 'The smoke has reduced and the air is good today.')
('17', 'positive', 'neutral', 'The repair was fast and the broken sensor now works.')
('62', 'positive', 'negative', 'The service is not not helpful, according to the caller.')

EXTRACTIVE SUMMARY COMPARISON
Important-word summary:
- Air feels clean and the monitoring service is helpful.
- The air is clean near the school and the staff are helpful.
- The new filter is helpful and the air is good.
- Smoke is heavy and the air is dirty.
- The factory smoke makes the air dirty and bad.
- The team was not helpful and the air is dirty.
- Wonderful, another clean-air d

## Result and Interpretation

Compare the sentiment labels with the reference labels. Review disagreements rather than treating the agreement percentage as proof that the classifier is fully reliable. Compare the two summary outputs, inspect the keyword coverage, and record the index timing and query plan. Use those observed results to support the production decision.